In [0]:
%run /Users/nethumgimsara605@gmail.com/ecommerce-lakehouse-databricks-repo/01-ingestion/setup-storage-connection

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count, month, year, date_format

In [0]:
silver_orders = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/orders/")
silver_products = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/products/")

print(f"Orders: {silver_orders.count()}")
print(f"Products: {silver_products.count()}")

In [0]:
orders_with_category = silver_orders.join(
    silver_products.select("product_id","category"),
    on="product_id",
    how="inner"
)

orders_with_category.show(5)
print(f"Joined rows: {orders_with_category.count()}")

In [0]:
orders_with_month = orders_with_category.withColumn(
    "order_year_month", date_format(col("order_timestamp"), "yyyy-MM")
)

orders_with_month.select("order_timestamp", "order_year_month").show(5)

In [0]:
gold_revenue_by_category_month = orders_with_month.groupBy("category", "order_year_month").agg(
    spark_sum("total_amount").alias("total_revenue"),
    count("order_id").alias("total_orders")
)

gold_revenue_by_category_month.show(5)

In [0]:
gold_revenue_by_category_month.write.format("delta").mode("overwrite").save("abfss://gold@ecommercelakehouse01.dfs.core.windows.net/revenue_by_category_month/")

In [0]:
gold_check = spark.read.format("delta").load("abfss://gold@ecommercelakehouse01.dfs.core.windows.net/revenue_by_category_month/")
gold_check.printSchema()
print(f"Total rows: {gold_check.count()}")
gold_check.orderBy("order_year_month", "category").show(20)